# Create Reports for Week 3.5 and Week 4

Run this notebook in Google Colab. It mounts Google Drive, reads result files from:

- `MyDrive/Thesis/Week 3.5`
- `MyDrive/Thesis/Week 4`

Then it saves new Markdown reports into:

`MyDrive/Thesis/Week 3.5 and Week 4 Reports`

The notebook searches recursively inside the Week 3.5 and Week 4 folders, so it works even if your result files are nested under run-specific folders.

In [ ]:
# Mount Google Drive when running in Colab.
from pathlib import Path
import csv
import json
import math
from datetime import datetime, timezone

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Not running in Colab, or Drive is already available. Continuing.')
    print(type(exc).__name__, exc)

DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_ROOT = DRIVE_ROOT / 'Thesis'

# Local fallback for testing outside Colab. In Colab, PROJECT_ROOT above is used.
if not PROJECT_ROOT.exists():
    cwd = Path.cwd()
    if (cwd / 'Week 3.5').exists() and (cwd / 'Week 4').exists():
        PROJECT_ROOT = cwd
    elif (cwd.parent / 'Week 3.5').exists() and (cwd.parent / 'Week 4').exists():
        PROJECT_ROOT = cwd.parent

WEEK35_DIR = PROJECT_ROOT / 'Week 3.5'
WEEK4_DIR = PROJECT_ROOT / 'Week 4'
REPORTS_DIR = PROJECT_ROOT / 'Week 3.5 and Week 4 Reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Week 3.5 folder:', WEEK35_DIR)
print('Week 4 folder:', WEEK4_DIR)
print('Reports folder:', REPORTS_DIR)

if not WEEK35_DIR.exists():
    raise FileNotFoundError(f'Missing Week 3.5 folder: {WEEK35_DIR}')
if not WEEK4_DIR.exists():
    raise FileNotFoundError(f'Missing Week 4 folder: {WEEK4_DIR}')

## Helper Functions

In [ ]:
def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))

def read_csv_dicts(path: Path):
    with path.open('r', encoding='utf-8', newline='') as handle:
        return list(csv.DictReader(handle))

def fmt_pct(value, digits=2):
    if value is None:
        return 'n/a'
    try:
        value = float(value)
    except Exception:
        return str(value)
    if math.isnan(value):
        return 'n/a'
    return f'{value:.{digits}f}%'

def fmt_pp(value, digits=2):
    value = float(value)
    sign = '+' if value > 0 else ''
    return f'{sign}{value:.{digits}f} pp'

def markdown_table(headers, rows):
    lines = ['| ' + ' | '.join(headers) + ' |']
    lines.append('|' + '|'.join(['---'] * len(headers)) + '|')
    for row in rows:
        lines.append('| ' + ' | '.join(str(item) for item in row) + ' |')
    return '\n'.join(lines)

def find_json_by_run_name(base_dir: Path, run_name_contains: str):
    matches = []
    for path in base_dir.rglob('metrics.json'):
        try:
            data = load_json(path)
        except Exception:
            continue
        run_name = str(data.get('run_name', ''))
        if run_name_contains in run_name:
            matches.append((path, data))
    if not matches:
        return None, None
    matches.sort(key=lambda item: len(str(item[0])))
    return matches[0]

def find_week35_metrics(base_dir: Path):
    path, data = find_json_by_run_name(base_dir, 'week3_5')
    if path is not None:
        return path, data
    # Fallback for archived reference runs without run_name.
    candidates = []
    for path in base_dir.rglob('metrics.json'):
        try:
            data = load_json(path)
        except Exception:
            continue
        if 'lora_after_training' in data and 'base_before_training' in data:
            candidates.append((path, data))
    if not candidates:
        raise FileNotFoundError(f'Could not find Week 3.5 metrics.json under {base_dir}')
    candidates.sort(key=lambda item: ('reference_successful_run' not in str(item[0]), len(str(item[0]))))
    return candidates[0]

def find_week4_metrics(base_dir: Path):
    path, data = find_json_by_run_name(base_dir, 'week4_gradient_ascent')
    if path is not None:
        return path, data
    candidates = []
    for path in base_dir.rglob('metrics.json'):
        try:
            data = load_json(path)
        except Exception:
            continue
        if 'before_unlearning' in data and 'after_unlearning' in data:
            candidates.append((path, data))
    if not candidates:
        raise FileNotFoundError(f'Could not find Week 4 unlearning metrics.json under {base_dir}')
    candidates.sort(key=lambda item: len(str(item[0])))
    return candidates[0]

def find_neighbor_file(metrics_path: Path, filename: str):
    same_folder = metrics_path.parent / filename
    if same_folder.exists():
        return same_folder
    matches = list(metrics_path.parents[1].rglob(filename)) if len(metrics_path.parents) > 1 else []
    return matches[0] if matches else None

def nested_metric_pct(metrics: dict, section: str, key: str):
    return 100.0 * float(metrics[section][key]['contains_value'])

## Locate and Load Result Files

In [ ]:
week35_metrics_path, week35_metrics = find_week35_metrics(WEEK35_DIR)
week4_metrics_path, week4_metrics = find_week4_metrics(WEEK4_DIR)

week35_percentage_path = find_neighbor_file(week35_metrics_path, 'percentage_summary.csv')
week4_percentage_path = find_neighbor_file(week4_metrics_path, 'percentage_summary.csv')
week4_history_path = find_neighbor_file(week4_metrics_path, 'unlearning_history.csv')

print('Week 3.5 metrics:', week35_metrics_path)
print('Week 3.5 percentage summary:', week35_percentage_path)
print('Week 4 metrics:', week4_metrics_path)
print('Week 4 percentage summary:', week4_percentage_path)
print('Week 4 unlearning history:', week4_history_path)

week35_percentage = read_csv_dicts(week35_percentage_path) if week35_percentage_path else []
week4_percentage = read_csv_dicts(week4_percentage_path) if week4_percentage_path else []
week4_history = read_csv_dicts(week4_history_path) if week4_history_path else []

## Build Week 3.5 Tables

In [ ]:
week35_rows = []

if isinstance(week35_metrics.get('lora_after_training', {}).get('forget_all'), dict):
    week35_rows = [
        ['Forget, all prompts', week35_metrics['base_before_training']['forget_all']['num_examples'], fmt_pct(nested_metric_pct(week35_metrics, 'base_before_training', 'forget_all')), fmt_pct(nested_metric_pct(week35_metrics, 'lora_after_training', 'forget_all'))],
        ['Retain, all prompts', week35_metrics['base_before_training']['retain_all']['num_examples'], fmt_pct(nested_metric_pct(week35_metrics, 'base_before_training', 'retain_all')), fmt_pct(nested_metric_pct(week35_metrics, 'lora_after_training', 'retain_all'))],
        ['Forget held-out paraphrases', week35_metrics['base_before_training']['forget_heldout_paraphrases']['num_examples'], fmt_pct(nested_metric_pct(week35_metrics, 'base_before_training', 'forget_heldout_paraphrases')), fmt_pct(nested_metric_pct(week35_metrics, 'lora_after_training', 'forget_heldout_paraphrases'))],
        ['Retain held-out paraphrases', week35_metrics['base_before_training']['retain_heldout_paraphrases']['num_examples'], fmt_pct(nested_metric_pct(week35_metrics, 'base_before_training', 'retain_heldout_paraphrases')), fmt_pct(nested_metric_pct(week35_metrics, 'lora_after_training', 'retain_heldout_paraphrases'))],
        ['Forget seen prompts', week35_metrics['base_before_training']['forget_seen_prompts']['num_examples'], fmt_pct(nested_metric_pct(week35_metrics, 'base_before_training', 'forget_seen_prompts')), fmt_pct(nested_metric_pct(week35_metrics, 'lora_after_training', 'forget_seen_prompts'))],
        ['Retain seen prompts', week35_metrics['base_before_training']['retain_seen_prompts']['num_examples'], fmt_pct(nested_metric_pct(week35_metrics, 'base_before_training', 'retain_seen_prompts')), fmt_pct(nested_metric_pct(week35_metrics, 'lora_after_training', 'retain_seen_prompts'))],
        ['General controls', week35_metrics.get('num_general_control_questions', 50), fmt_pct(week35_metrics['general_control']['base_before_training_contains_value_percentage']), fmt_pct(week35_metrics['general_control']['lora_after_training_contains_value_percentage'])],
    ]
else:
    before = week35_metrics['base_before_training']
    after = week35_metrics['lora_after_training']
    week35_rows = [
        ['Synthetic forget', 300, fmt_pct(before.get('synthetic_forget_contains_value_percentage')), fmt_pct(after.get('synthetic_forget_contains_value_percentage'))],
        ['Synthetic retain', 1200, fmt_pct(before.get('synthetic_retain_contains_value_percentage')), fmt_pct(after.get('synthetic_retain_contains_value_percentage'))],
        ['General controls', week35_metrics.get('num_general_control_questions', 50), fmt_pct(before.get('general_control_contains_value_percentage')), fmt_pct(after.get('general_control_contains_value_percentage'))],
    ]

week35_table_md = markdown_table(['Evaluation group', 'Examples', 'Base before training', 'LoRA after training'], week35_rows)
print(week35_table_md)

## Build Week 4 Tables

In [ ]:
before = week4_metrics['before_unlearning']
after = week4_metrics['after_unlearning']

week4_rows = [
    ['Forget accuracy, all', fmt_pct(before['forget_all']), fmt_pct(after['forget_all']), fmt_pp(after['forget_all'] - before['forget_all'])],
    ['Forget held-out paraphrases', fmt_pct(before['forget_heldout_paraphrases']), fmt_pct(after['forget_heldout_paraphrases']), fmt_pp(after['forget_heldout_paraphrases'] - before['forget_heldout_paraphrases'])],
    ['Retain accuracy, all', fmt_pct(before['retain_all']), fmt_pct(after['retain_all']), fmt_pp(after['retain_all'] - before['retain_all'])],
    ['Retain held-out paraphrases', fmt_pct(before['retain_heldout_paraphrases']), fmt_pct(after['retain_heldout_paraphrases']), fmt_pp(after['retain_heldout_paraphrases'] - before['retain_heldout_paraphrases'])],
    ['General controls', fmt_pct(before['general']), fmt_pct(after['general']), fmt_pp(after['general'] - before['general'])],
]
week4_table_md = markdown_table(['Metric', 'Before unlearning', 'After unlearning', 'Change'], week4_rows)
print(week4_table_md)

history_table_md = ''
if week4_history:
    history_rows = []
    for row in week4_history:
        history_rows.append([
            int(float(row['epoch'])),
            fmt_pct(row['forget_train_percentage']),
            fmt_pct(row['retain_train_sample_percentage']),
            'Yes' if row['retain_eligible'].strip().lower() == 'true' else 'No',
            f"{float(row['selection_score']):.2f}",
        ])
    history_table_md = markdown_table(['Epoch', 'Forget train accuracy', 'Retain sample accuracy', 'Eligible', 'Selection score'], history_rows)
    print('\n' + history_table_md)

## Write Reports to Google Drive

In [ ]:
generated_at = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')

week35_report = f"""# Week 3.5 Report: High-Accuracy Learned Baseline

Generated on {generated_at} from files inside `Week 3.5`.

## Purpose

Week 3.5 created the learned model state used as the starting point for unlearning. The goal was to confirm that the model learned the synthetic forget and retain facts before Week 4 attempted removal.

## Source Files

- Metrics: `{week35_metrics_path.relative_to(PROJECT_ROOT)}`
- Percentage summary: `{week35_percentage_path.relative_to(PROJECT_ROOT) if week35_percentage_path else 'not found'}`

## Results

{week35_table_md}

## Interpretation

The Week 3.5 model shows strong synthetic-fact learning. This is important because Week 4 unlearning is only meaningful if the target facts were actually learned first. General-control performance should be interpreted cautiously because earlier contains-value scoring can overcount short numeric answers.
"""

history_section = f"\n## Checkpoint Selection\n\n{history_table_md}\n" if history_table_md else ''
week4_report = f"""# Week 4 Report: Gradient-Ascent Unlearning

Generated on {generated_at} from files inside `Week 4`.

## Purpose

Week 4 tested whether gradient-ascent unlearning could reduce the model's recall of the designated forget facts while preserving retain facts and general behavior.

## Source Files

- Metrics: `{week4_metrics_path.relative_to(PROJECT_ROOT)}`
- Percentage summary: `{week4_percentage_path.relative_to(PROJECT_ROOT) if week4_percentage_path else 'not found'}`
- Unlearning history: `{week4_history_path.relative_to(PROJECT_ROOT) if week4_history_path else 'not found'}`

## Configuration

- Run name: `{week4_metrics.get('run_name', 'unknown')}`
- Method: {week4_metrics.get('method', 'unknown')}
- Selected epoch: {week4_metrics.get('selected_epoch', 'unknown')}
- Learning rate: {week4_metrics.get('training', {}).get('learning_rate', 'unknown')}
- Retain weight: {week4_metrics.get('training', {}).get('retain_weight', 'unknown')}

## Results

{week4_table_md}
{history_section}
## Interpretation

Gradient ascent substantially reduced forget accuracy, including on held-out paraphrases, but retain accuracy also decreased. The result supports partial selective suppression with a utility trade-off rather than complete knowledge deletion.
"""

combined_report = f"""# Week 3.5 and Week 4 Combined Result Summary

Generated on {generated_at}.

## Week 3.5 Learning Result

{week35_table_md}

## Week 4 Unlearning Result

{week4_table_md}

## Main Takeaway

Week 3.5 established a learned synthetic-fact adapter. Week 4 reduced forget accuracy from {fmt_pct(before['forget_all'])} to {fmt_pct(after['forget_all'])}, but retain accuracy also changed from {fmt_pct(before['retain_all'])} to {fmt_pct(after['retain_all'])}. This means the current result is best described as partial unlearning with collateral damage.
"""

source_manifest = {
    'generated_at': generated_at,
    'project_root': str(PROJECT_ROOT),
    'reports_dir': str(REPORTS_DIR),
    'week35_metrics': str(week35_metrics_path),
    'week35_percentage_summary': str(week35_percentage_path) if week35_percentage_path else None,
    'week4_metrics': str(week4_metrics_path),
    'week4_percentage_summary': str(week4_percentage_path) if week4_percentage_path else None,
    'week4_unlearning_history': str(week4_history_path) if week4_history_path else None,
}

outputs = {
    REPORTS_DIR / 'week_3_5_report.md': week35_report,
    REPORTS_DIR / 'week_4_report.md': week4_report,
    REPORTS_DIR / 'week_3_5_and_week_4_summary.md': combined_report,
}

for path, text in outputs.items():
    path.write_text(text.strip() + '\n', encoding='utf-8')
    print('Wrote', path)

(REPORTS_DIR / 'source_files.json').write_text(json.dumps(source_manifest, indent=2), encoding='utf-8')
print('Wrote', REPORTS_DIR / 'source_files.json')

## Preview Report Files

In [ ]:
for path in sorted(REPORTS_DIR.glob('*')):
    print(path.name)